In [ ]:
bad_mask = np.zeros(n_stations, dtype=bool)
bad_mask[BAD_IDX] = True

x = [tem.data['E'][i] for i in np.flatnonzero(~bad_mask)]
y = [tem.data['N'][i] for i in np.flatnonzero(~bad_mask)]
elev = [tem.data['Elevation'][i] for i in np.flatnonzero(~bad_mask)]
rhos = rho_sections[~bad_mask]
thicks = thicknesses*np.ones_like(rhos[:,:-1])
residual = rms_stations[~bad_mask]
doi_con = np.ones_like(rms_stations[~bad_mask]) * 100
doi_standard = np.ones_like(rms_stations[~bad_mask]) * 100
line_num = [int(tem.data['Line'][i]) for i in np.flatnonzero(~bad_mask)]

mod_dict = {
    'x':            x,
    'y':            y,
    'elev':         elev,
    'rhos':         rhos,
    'depths':       depths,   # DEP_BOT
    'doi_con':      doi_con,
    'doi_standard': doi_standard,
    'residual':     residual,
    'line_num':     line_num,
    'instrument':   'tTEM',
    'epsg':         'epsg:25832',
}

import re

def writeXYZ(mod_dict, xyz_path, instrument=None, epsg=None,
             model_type='tTEM'):
    """
    Write a TEM model dictionary to an Aarhus Workbench-style .xyz file.

    The output is readable by `loadXYZ` (round-trip compatible).

    Parameters
    ----------
    mod_dict : DICT
        A model dict with keys: 'x', 'y', 'elev', 'rhos', 'depths',
        'doi_con', 'doi_standard', 'residual', 'line_num'.
    xyz_path : STRING
        Output path for the .xyz file.
    instrument : STRING, optional
        Instrument string (e.g. 'tTEM'). If None, tries mod_dict['instrument'].
    epsg : STRING or INT, optional
        EPSG code (e.g. 'epsg:25832' or 25832). If None, tries
        mod_dict['epsg'].
    model_type : STRING
        Kept for API symmetry with loadXYZ; not used in writing.
    """

    # ---- Resolve metadata -------------------------------------------------
    if instrument is None:
        instrument = mod_dict.get('instrument', model_type)
    if epsg is None:
        epsg = mod_dict.get('epsg', 'epsg:25832')

    # Normalise epsg to 'epsg:XXXX'
    epsg_str = str(epsg)
    if not epsg_str.lower().startswith('epsg:'):
        epsg_str = 'epsg:' + epsg_str
    epsg_code = re.search(r'(\d+)', epsg_str).group(1)

    # ---- Extract arrays ---------------------------------------------------
    x            = np.asarray(mod_dict['x'])
    y            = np.asarray(mod_dict['y'])
    elev         = np.asarray(mod_dict['elev'])
    rhos         = np.asarray(mod_dict['rhos'])       # (n_soundings, n_layers)
    depths       = np.asarray(mod_dict['depths'])     # (n_soundings, n_layers)
    doi_con      = np.asarray(mod_dict['doi_con'])
    doi_standard = np.asarray(mod_dict['doi_standard'])
    residual     = np.asarray(mod_dict['residual'])
    line_num     = np.asarray(mod_dict['line_num']).astype(int)

    n_soundings, n_layers = rhos.shape
    if depths.shape != rhos.shape:
        raise ValueError(
            f"rhos shape {rhos.shape} != depths shape {depths.shape}"
        )

    # ---- Build column names ----------------------------------------------
    rho_cols = [f"RHO{i+1}"         for i in range(n_layers)]
    rho_std  = [f"RHO{i+1}_STD"     for i in range(n_layers)]
    dep_cols = [f"DEP_BOT{i+1}"     for i in range(n_layers)]
    dep_std  = [f"DEP_BOT{i+1}_STD" for i in range(n_layers)]

    base_cols   = ['LINE_NO', 'UTMX', 'UTMY', 'ELEVATION']
    tail_cols   = ['DOI_CONSERVATIVE', 'DOI_STANDARD', 'RESDATA']

    header_cols = base_cols + rho_cols + rho_std + dep_cols + dep_std + tail_cols

    # ---- Assemble the data matrix ----------------------------------------
    data = np.full((n_soundings, len(header_cols)), np.nan, dtype=float)

    def col_slice(cols):
        start = header_cols.index(cols[0])
        return slice(start, start + len(cols))

    data[:, header_cols.index('LINE_NO')]   = line_num
    data[:, header_cols.index('UTMX')]      = x
    data[:, header_cols.index('UTMY')]      = y
    data[:, header_cols.index('ELEVATION')] = elev

    data[:, col_slice(rho_cols)] = rhos
    data[:, col_slice(dep_cols)] = depths

    data[:, header_cols.index('DOI_CONSERVATIVE')] = doi_con
    data[:, header_cols.index('DOI_STANDARD')]     = doi_standard
    data[:, header_cols.index('RESDATA')]          = residual

    # ---- Write the file ---------------------------------------------------
    with open(xyz_path, 'w') as f:
        f.write("Aarhus Workbench XYZ export\n")
        f.write("DATA TYPE\n")
        f.write(f"{instrument}/DT\n")
        f.write("COORDINATE SYSTEM\n")
        f.write(f"epsg:{epsg_code}\n")
        f.write("/ LINE_NO " + " ".join(header_cols[1:]) + "\n")

        for row in data:
            f.write(" ".join(f"{v:.6g}" for v in row) + "\n")

    print(f"Wrote {n_soundings} soundings x {n_layers} layers -> {xyz_path}")

xyz_path = "output.xyz"

writeXYZ(mod_dict, xyz_path, instrument=None, epsg='25832',
             model_type='sTEM')